In [9]:
import json

with open("baseline_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

In [17]:
!pip -q install transformers accelerate

In [18]:
import json
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)
model.eval()

In [10]:
DATA_PATH = "data/qa_set.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    qa_set = json.load(f)

In [11]:
RETRY_INSTRUCTIONS = """
Your previous output was invalid.

Return ONLY valid JSON using this schema:

{
  "answer": string,
  "refused": boolean,
  "confidence": number
}

Rules:
- Do not include explanations
- Do not include markdown
- Return ONLY JSON
""".strip()

In [27]:
def llm_generate_json(question, seed=0, temperature=0.7, top_p=0.9, max_new_tokens=120):
    set_seed(seed)
    prompt = f"{RETRY_INSTRUCTIONS}\n\nQuestion: {question}"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id
    )

    # ✅ Only decode newly generated tokens (exclude the prompt)
    new_tokens = out[0][input_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return text

In [29]:
def extract_first_json_block(text: str):
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    return match.group(0) if match else None

In [21]:
def retry_fix(previous_output):
    prompt = f"""
{RETRY_INSTRUCTIONS}

Previous output:
{previous_output}

Corrected JSON:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(out[0], skip_special_tokens=True)

    return text

In [22]:
def generate_with_retry(question, max_retries=2):

    raw = llm_generate_json(question)

    block = extract_first_json_block(raw)

    try:
        parsed = json.loads(block)
        ok, errors = strict_validate(parsed)
    except:
        ok = False
        errors = ["json_parse_error"]

    retries = 0

    while not ok and retries < max_retries:

        fixed = retry_fix(raw)

        block = extract_first_json_block(fixed)

        try:
            parsed = json.loads(block)
            ok, errors = strict_validate(parsed)
            raw = fixed
        except:
            ok = False

        retries += 1

    return raw, ok, retries

In [ ]:
retry_results = []

for ex in qa_set:

    output, ok, retries = generate_with_retry(ex["question"])

    retry_results.append({
        "question": ex["question"],
        "answerable": ex["answerable"],
        "final_output": output,
        "valid": ok,
        "retries": retries
    })

In [31]:
baseline_valid = sum(r["json_valid"] for r in results)
retry_valid = sum(r["valid"] for r in retry_results)

print("Baseline JSON valid:", baseline_valid)
print("After retry JSON valid:", retry_valid)

Baseline JSON valid: 18
After retry JSON valid: 0


In [32]:
total_retries = sum(r["retries"] for r in retry_results)

print("Total retries:", total_retries)
print("Average retries:", total_retries / len(retry_results))

Total retries: 40
Average retries: 2.0


In [33]:
import json

with open("retry_results.json", "w", encoding="utf-8") as f:
    json.dump(retry_results, f, indent=2)